# Fire Hazard Environment Check - Both Experiments
This notebook prepares and validates both experiment datasets without training. It keeps your original dataset unchanged, creates a cleaned segmentation copy by converting 5-value box lines into rectangle polygons, creates a detection copy, validates both, previews labels, and writes a final report.

## 1. Project Configuration

In [ ]:
RUN_MODE = 'both'  # 'segmentation', 'detection', or 'both'
AUTO_CREATE_CLEAN_SEGMENTATION_DATASET = True
AUTO_CREATE_DETECTION_DATASET = True
OVERWRITE_DERIVED_DATASETS = True

SEG_SOURCE_DATASET_ROOT = '/content/drive/MyDrive/fire_hazard_dataset/Fire Hazard YOLO26'
SEG_CLEAN_DATASET_ROOT = '/content/drive/MyDrive/fire_hazard_dataset/Fire Hazard YOLO26 Segmentation Clean'
DET_DATASET_ROOT = '/content/drive/MyDrive/fire_hazard_dataset/Fire Hazard YOLO26 Detection'
OUTPUT_ROOT = '/content/drive/MyDrive/fire_hazard_experiment_outputs'

SEG_SOURCE_DATA_YAML = f'{SEG_SOURCE_DATASET_ROOT}/data.yaml'
SEG_CLEAN_DATA_YAML = f'{SEG_CLEAN_DATASET_ROOT}/data.yaml'
DET_DATA_YAML = f'{DET_DATASET_ROOT}/data.yaml'
EXPECTED_IMAGE_COUNT = 23
seed = 42
checks = {}
reports = {}
print('SEG_SOURCE_DATASET_ROOT:', SEG_SOURCE_DATASET_ROOT)
print('SEG_CLEAN_DATASET_ROOT:', SEG_CLEAN_DATASET_ROOT)
print('DET_DATASET_ROOT:', DET_DATASET_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 2. Mount Google Drive

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
SEG_SOURCE_DATASET_ROOT_PATH = Path(SEG_SOURCE_DATASET_ROOT)
SEG_CLEAN_DATASET_ROOT_PATH = Path(SEG_CLEAN_DATASET_ROOT)
DET_DATASET_ROOT_PATH = Path(DET_DATASET_ROOT)
OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT)
ENV_CHECK_ROOT = OUTPUT_ROOT_PATH / 'environment_check'
checks['seg_source_dataset_root'] = SEG_SOURCE_DATASET_ROOT_PATH.exists()
print('Source segmentation dataset:', 'FOUND' if checks['seg_source_dataset_root'] else 'MISSING')
if not checks['seg_source_dataset_root']:
    print('Expected:', SEG_SOURCE_DATASET_ROOT)

## 3. Clone or Locate Repository

In [ ]:
import subprocess, sys
REPO_URL = 'https://github.com/rachataktn/fire-hazard-spatial.git'
REPO_ROOT = Path('/content/fire-hazard-spatial')
if not (REPO_ROOT / 'scripts' / 'check_dataset.py').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=False)
checks['repository'] = (REPO_ROOT / 'scripts' / 'check_dataset.py').exists()
print('Repository:', 'PASS' if checks['repository'] else 'FAIL')
print('Repository root:', REPO_ROOT)

## 4. GPU and Packages

In [ ]:
import importlib, platform, subprocess, sys
print('Python:', sys.version)
print('Operating system:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA version:', torch.version.cuda)
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))
    else:
        print('WARNING: CUDA GPU is not available. Runtime -> Change runtime type -> GPU')
    checks['gpu'] = torch.cuda.is_available()
except Exception as exc:
    print('PyTorch check failed:', exc)
    checks['gpu'] = False
packages = {'ultralytics':'ultralytics','numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','Pillow':'PIL','PyYAML':'yaml','opencv-python-headless':'cv2'}
missing = []
for package, import_name in packages.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing.append(package)
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import ultralytics, numpy as np, pandas as pd
print('Ultralytics version:', ultralytics.__version__)
print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)
checks['packages'] = True

## 5. YOLO26 Availability Inspection

In [ ]:
import ultralytics
root = Path(ultralytics.__file__).resolve().parent
def package_mentions(term):
    hits = []
    for path in root.rglob('*.py'):
        try:
            text = path.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        if term.lower() in text.lower():
            hits.append(str(path.relative_to(root)))
    return hits[:25]
mentions_yolo26 = package_mentions('yolo26')
checks['yolo26_detection'] = bool(mentions_yolo26)
checks['yolo26_segmentation'] = bool(mentions_yolo26) and bool(package_mentions('segment'))
checks['yolo26_depth'] = bool(mentions_yolo26) and bool(package_mentions('depth'))
print('YOLO26 detection:', 'PASS' if checks['yolo26_detection'] else 'FAIL / not found in installed package text')
print('YOLO26 segmentation:', 'PASS' if checks['yolo26_segmentation'] else 'FAIL / requires official API verification')
print('YOLO26-Depth:', 'PASS' if checks['yolo26_depth'] else 'FAIL / requires official API verification')
print('Candidate segmentation checkpoint: yolo26n-seg.pt')
print('Candidate detection checkpoint: yolo26n.pt')
print('Candidate depth checkpoint: yolo26n-depth.pt or official equivalent')

## 6. Create Clean Segmentation Dataset

In [ ]:
import shutil, yaml
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
def bbox_to_rectangle_polygon(fields):
    cid, x, y, w, h = fields
    x, y, w, h = float(x), float(y), float(w), float(h)
    x_min = max(0.0, x - w / 2); y_min = max(0.0, y - h / 2)
    x_max = min(1.0, x + w / 2); y_max = min(1.0, y + h / 2)
    coords = [x_min, y_min, x_max, y_min, x_max, y_max, x_min, y_max]
    return ' '.join([cid, *[f'{v:.10f}' for v in coords]])
def create_clean_segmentation(src_root, clean_root, src_yaml):
    src_root, clean_root, src_yaml = Path(src_root), Path(clean_root), Path(src_yaml)
    if clean_root.exists() and OVERWRITE_DERIVED_DATASETS:
        shutil.rmtree(clean_root)
    clean_root.mkdir(parents=True, exist_ok=True)
    converted = 0
    for split in ('train', 'valid', 'test'):
        for sub in ('images', 'labels'):
            (clean_root / split / sub).mkdir(parents=True, exist_ok=True)
        src_images = src_root / split / 'images'
        src_labels = src_root / split / 'labels'
        if src_images.exists():
            for image in src_images.iterdir():
                if image.is_file() and image.suffix.lower() in IMAGE_EXTS:
                    shutil.copy2(image, clean_root / split / 'images' / image.name)
        if src_labels.exists():
            for label in src_labels.glob('*.txt'):
                out_lines = []
                for raw in label.read_text(encoding='utf-8', errors='replace').splitlines():
                    fields = raw.strip().split()
                    if not fields:
                        continue
                    if len(fields) == 5:
                        out_lines.append(bbox_to_rectangle_polygon(fields)); converted += 1
                    else:
                        out_lines.append(' '.join(fields))
                (clean_root / split / 'labels' / label.name).write_text('\n'.join(out_lines) + ('\n' if out_lines else ''), encoding='utf-8')
    y = yaml.safe_load(src_yaml.read_text(encoding='utf-8')) or {}
    y['train'] = 'train/images'; y['val'] = 'valid/images'
    if (src_root / 'test').exists(): y['test'] = 'test/images'
    y.pop('path', None)
    (clean_root / 'data.yaml').write_text(yaml.safe_dump(y, sort_keys=False), encoding='utf-8')
    return converted
if AUTO_CREATE_CLEAN_SEGMENTATION_DATASET:
    converted = create_clean_segmentation(SEG_SOURCE_DATASET_ROOT, SEG_CLEAN_DATASET_ROOT, SEG_SOURCE_DATA_YAML)
    print('Clean segmentation dataset written to:', SEG_CLEAN_DATASET_ROOT)
    print('5-value box lines converted to rectangle polygons:', converted)
else:
    print('Clean segmentation dataset creation skipped.')

## 7. Create Detection Dataset

In [ ]:
if RUN_MODE in {'detection', 'both'} and AUTO_CREATE_DETECTION_DATASET and checks.get('repository'):
    cmd = [sys.executable, str(REPO_ROOT / 'scripts' / 'check_dataset.py'), '--dataset-root', SEG_CLEAN_DATASET_ROOT, '--data-yaml', SEG_CLEAN_DATA_YAML, '--task-mode', 'segmentation', '--expected-images', str(EXPECTED_IMAGE_COUNT), '--convert-detection-root', DET_DATASET_ROOT]
    if OVERWRITE_DERIVED_DATASETS:
        cmd.append('--overwrite-converted')
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
else:
    print('Detection dataset conversion skipped.')

## 8. Validate Clean Segmentation and Detection Datasets

In [ ]:
import json
def run_validator(name, root, yaml_path, task_mode):
    report_json = Path(f'/tmp/{name}_dataset_report.json')
    cmd = [sys.executable, str(REPO_ROOT / 'scripts' / 'check_dataset.py'), '--dataset-root', root, '--data-yaml', yaml_path, '--task-mode', task_mode, '--expected-images', str(EXPECTED_IMAGE_COUNT), '--json-output', str(report_json)]
    result = subprocess.run(cmd, text=True, capture_output=True)
    print('\n' + '=' * 60)
    print(name.upper(), task_mode.upper(), 'VALIDATION')
    print('=' * 60)
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    report = json.loads(report_json.read_text(encoding='utf-8')) if report_json.exists() else {'status': 'INVALID'}
    reports[name] = report
    checks[f'{name}_valid'] = report.get('status') == 'VALID'
if RUN_MODE in {'segmentation', 'both'}:
    run_validator('segmentation', SEG_CLEAN_DATASET_ROOT, SEG_CLEAN_DATA_YAML, 'segmentation')
if RUN_MODE in {'detection', 'both'}:
    run_validator('detection', DET_DATASET_ROOT, DET_DATA_YAML, 'detection')

## 9. Test Writing Outputs to Google Drive

In [ ]:
try:
    ENV_CHECK_ROOT.mkdir(parents=True, exist_ok=True)
    test_file = ENV_CHECK_ROOT / 'environment_test.txt'
    expected = 'Google Drive output write test successful.'
    test_file.write_text(expected, encoding='utf-8')
    checks['drive_output'] = test_file.read_text(encoding='utf-8') == expected
    print('Google Drive output write test:', 'PASS' if checks['drive_output'] else 'FAIL')
except Exception as exc:
    checks['drive_output'] = False
    print('Google Drive output write test: FAIL', exc)

## 10. Visual Annotation Preview

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
def label_for_image(image_path):
    parts = list(image_path.parts)
    for i, part in enumerate(parts):
        if part == 'images': parts[i] = 'labels'; return Path(*parts).with_suffix('.txt')
    return image_path.with_suffix('.txt')
def draw_preview(dataset_root, report, task_mode, out_dir):
    dataset_root = Path(dataset_root); out_dir.mkdir(parents=True, exist_ok=True)
    class_names = {int(k): v for k, v in report.get('classes', {}).items()} if report else {}
    images = sorted([p for p in dataset_root.rglob('*') if p.suffix.lower() in IMAGE_EXTS]) if dataset_root.exists() else []
    random.seed(seed); sample_images = random.sample(images, min(6, len(images))) if images else []
    for idx, image_path in enumerate(sample_images, 1):
        im = Image.open(image_path).convert('RGB'); w, h = im.size
        fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(im); ax.axis('off')
        label_path = label_for_image(image_path)
        if label_path.exists():
            for raw in label_path.read_text(encoding='utf-8', errors='replace').splitlines():
                vals = raw.split()
                if task_mode == 'detection' and len(vals) == 5:
                    cid, x, y, bw, bh = map(float, vals); x1, y1 = (x - bw / 2) * w, (y - bh / 2) * h
                    ax.add_patch(patches.Rectangle((x1, y1), bw*w, bh*h, fill=False, linewidth=2, edgecolor='lime'))
                    ax.text(x1, max(0, y1-4), class_names.get(int(cid), str(int(cid))), color='black', backgroundcolor='lime', fontsize=9)
                elif task_mode == 'segmentation' and len(vals) >= 7 and (len(vals)-1) % 2 == 0:
                    cid = int(float(vals[0])); coords = list(map(float, vals[1:]))
                    points = [(coords[i]*w, coords[i+1]*h) for i in range(0, len(coords), 2)]
                    ax.add_patch(patches.Polygon(points, fill=False, linewidth=2, edgecolor='cyan'))
                    ax.text(points[0][0], max(0, points[0][1]-4), class_names.get(cid, str(cid)), color='black', backgroundcolor='cyan', fontsize=9)
        fig.savefig(out_dir / f'preview_{idx:02d}.png', bbox_inches='tight', dpi=150)
        plt.show(); plt.close(fig)
if 'segmentation' in reports:
    draw_preview(SEG_CLEAN_DATASET_ROOT, reports['segmentation'], 'segmentation', ENV_CHECK_ROOT / 'annotation_preview_segmentation')
if 'detection' in reports:
    draw_preview(DET_DATASET_ROOT, reports['detection'], 'detection', ENV_CHECK_ROOT / 'annotation_preview_detection')

## 11. Optional One-Image Inference Smoke Tests

In [ ]:
RUN_DETECTION_SMOKE_TEST = False
RUN_SEGMENTATION_SMOKE_TEST = False
RUN_DEPTH_SMOKE_TEST = False
DETECTION_MODEL_NAME = 'yolo26n.pt'
SEGMENTATION_MODEL_NAME = 'yolo26n-seg.pt'
DEPTH_MODEL_NAME = 'yolo26n-depth.pt'
checks['detection_inference'] = 'NOT RUN'
checks['segmentation_inference'] = 'NOT RUN'
checks['depth_inference'] = 'NOT RUN'
print('Smoke tests are off by default. Enable only after checkpoint names are confirmed.')

## 12. Final Environment Report

In [ ]:
def pf(value): return 'PASS' if value else 'FAIL'
wanted = [RUN_MODE] if RUN_MODE in {'segmentation', 'detection'} else ['segmentation', 'detection']
dataset_ready = all(checks.get(f'{name}_valid', False) for name in wanted)
ready = all([checks.get('repository', False), checks.get('seg_source_dataset_root', False), checks.get('gpu', False), checks.get('packages', False), checks.get('drive_output', False), dataset_ready])
lines = ['===========================================','FIRE HAZARD EXPERIMENT - ENVIRONMENT CHECK','===========================================','',f'Run mode: {RUN_MODE}',f'Repository: {pf(checks.get("repository", False))}',f'Google Drive: {pf(Path("/content/drive").exists())}',f'Source segmentation dataset root: {pf(checks.get("seg_source_dataset_root", False))}',f'Clean segmentation dataset root: {pf(Path(SEG_CLEAN_DATASET_ROOT).exists())}',f'Detection dataset root: {pf(Path(DET_DATASET_ROOT).exists())}',f'Segmentation validation: {pf(checks.get("segmentation_valid", False)) if "segmentation" in wanted else "NOT RUN"}',f'Detection validation: {pf(checks.get("detection_valid", False)) if "detection" in wanted else "NOT RUN"}',f'GPU: {pf(checks.get("gpu", False))}',f'PyTorch: {torch.__version__ if "torch" in globals() else "unknown"}',f'Ultralytics: {ultralytics.__version__ if "ultralytics" in globals() else "unknown"}',f'YOLO26 detection: {pf(checks.get("yolo26_detection", False))}',f'YOLO26 segmentation: {pf(checks.get("yolo26_segmentation", False))}',f'YOLO26-Depth: {pf(checks.get("yolo26_depth", False))}',f'Google Drive output: {pf(checks.get("drive_output", False))}',f'Single-image detection inference: {checks.get("detection_inference", "NOT RUN")}',f'Single-image segmentation inference: {checks.get("segmentation_inference", "NOT RUN")}',f'Single-image depth inference: {checks.get("depth_inference", "NOT RUN")}', '']
for name, report in reports.items():
    lines += [f'{name.title()} images: {report.get("total_images", "unknown")}', f'{name.title()} labels: {report.get("total_labels", "unknown")}', f'{name.title()} annotations: {report.get("total_annotations", "unknown")}', '']
if not ready:
    lines.append('Fix before full experiment:')
    if not checks.get('repository', False): lines.append('- Clone the GitHub repository so scripts/check_dataset.py is available.')
    if not checks.get('seg_source_dataset_root', False): lines.append('- Correct SEG_SOURCE_DATASET_ROOT.')
    if not dataset_ready: lines.append('- Review validation output above for invalid dataset paths or labels.')
    if not checks.get('gpu', False): lines.append('- Enable GPU: Runtime -> Change runtime type -> GPU.')
    if not checks.get('drive_output', False): lines.append('- Check OUTPUT_ROOT permissions and Drive storage.')
    lines.append('')
lines += [f'READY FOR FULL EXPERIMENT: {"YES" if ready else "NO"}', '===========================================']
final_report = '\n'.join(lines)
print(final_report)
ENV_CHECK_ROOT.mkdir(parents=True, exist_ok=True)
(ENV_CHECK_ROOT / 'environment_report.txt').write_text(final_report, encoding='utf-8')
print('\nSaved report to:', ENV_CHECK_ROOT / 'environment_report.txt')